# ViT-B/16 Multilabel UCMerced — Stratified vs Random Split

## Hipótesis
Los modelos ViT pre-entrenados (ImageNet, ~86 M parámetros) son suficientemente robustos al desbalance de clases como para **no** degradarse significativamente cuando el split train/val es aleatorio en lugar de estratificado.
- **H0 (hipótesis verdadera):** Δ métricas entre ambos modelos < 2 pp → el pre-entrenamiento compensa el desbalance.
- **H1 (hipótesis falsa):** Δ métricas ≥ 2 pp → el split estratificado sigue siendo necesario.

Ambos modelos son ViT-B/16 idénticos (misma seed, mismos pesos iniciales), solo difiere el tipo de split de train/val.

## 1. Setup — Drive, dependencias y repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())

# Install dependencies
!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image wandb iterative-stratification torchmetrics

In [ ]:
# Run this everytime you update something in the repo!
REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("Working in:", os.getcwd())

## 2. Descarga del dataset UCMerced

In [ ]:
import zipfile, subprocess, shutil

if not os.path.exists('ucmdata'):
    print("Cloning ucmdata repo and extracting images...")
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')
    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zf:
        zf.extractall('UCMImages')
    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
    os.chdir(PROJECT_DIR)
    print("Dataset ready.")
else:
    print("Dataset already present.")

## 3. Importaciones

In [ ]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as tvm
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Import from utils — reload to pick up any latest changes
import utils
importlib.reload(utils)
from utils import (
    build_dataloaders,
    LightningModuleMultilabel,
    compute_test_metrics,
    append_metrics_to_csv,
    plot_training_curves,
    plot_per_class_metrics,
)

L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
Path("outputs/checkpoints").mkdir(parents=True, exist_ok=True)
Path("outputs/logs").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Hiperparámetros

In [ ]:
PRETRAINED_MODEL      = "vit_b_16"
MAX_EPOCHS            = 30
EARLYSTOPPING_EPOCHS  = 7
LR                    = 3e-5    # Lower LR recommended for ViT fine-tuning
WEIGHT_DECAY          = 5e-2    # Standard weight decay for ViT (Dosovitskiy et al.)
THRESHOLD             = 0.5
SEED                  = 42
BATCH_SIZE            = 32
NUM_WORKERS           = 2

print(f"Model:        {PRETRAINED_MODEL}")
print(f"Max epochs:   {MAX_EPOCHS}  |  Early stopping: {EARLYSTOPPING_EPOCHS}")
print(f"LR:           {LR}  |  Weight decay: {WEIGHT_DECAY}")
print(f"Batch size:   {BATCH_SIZE}  |  Threshold: {THRESHOLD}")

## 5. Dataloaders — Modelo A (Stratified) y Modelo B (Random)

Se construyen dos sets de dataloaders con `build_dataloaders()` desde `utils.py`.  
El argumento `stratified` controla el tipo de split:
- `stratified=True` → `MultilabelStratifiedShuffleSplit` (preserva distribución de clases)
- `stratified=False` → shuffle aleatorio con la misma seed

In [ ]:
# ── Modelo A: Stratified split ──────────────────────────────────────────────
print("=" * 55)
print("  DATALOADER A — STRATIFIED SPLIT")
print("=" * 55)

train_loader_A, val_loader_A, test_loader_A, classes, pos_w_A = build_dataloaders(
    root_dir    = "ucmdata",
    label_file  = "LandUse_Multilabeled.txt",
    image_size  = (224, 224),
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    val_frac    = 0.15,
    test_frac   = 0.15,
    seed        = SEED,
    image_ext   = ".tif",
    stratified  = True,
)
NUM_CLASSES = len(classes)
print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_loader_A.dataset)} | Val: {len(val_loader_A.dataset)} | Test: {len(test_loader_A.dataset)}")

In [ ]:
# ── Modelo B: Random split ───────────────────────────────────────────────────
print("=" * 55)
print("  DATALOADER B — RANDOM SPLIT")
print("=" * 55)

train_loader_B, val_loader_B, test_loader_B, _, pos_w_B = build_dataloaders(
    root_dir    = "ucmdata",
    label_file  = "LandUse_Multilabeled.txt",
    image_size  = (224, 224),
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    val_frac    = 0.15,
    test_frac   = 0.15,
    seed        = SEED,
    image_ext   = ".tif",
    stratified  = False,
)
print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_loader_B.dataset)} | Val: {len(val_loader_B.dataset)} | Test: {len(test_loader_B.dataset)}")

## 6. Arquitectura ViT-B/16

Se reemplaza únicamente la cabeza de clasificación (`model.heads.head`) por una capa `Linear(768 → 17)`.  
Todos los pesos del transformer permanecen entrenables (fine-tuning completo).

In [ ]:
def build_vit_b16(num_classes: int):
    """ViT-B/16 pre-trained on ImageNet1K with new multilabel classification head."""
    weights = tvm.ViT_B_16_Weights.IMAGENET1K_V1
    model   = tvm.vit_b_16(weights=weights)
    in_feat = model.heads.head.in_features   # 768
    model.heads.head = nn.Linear(in_feat, num_classes)
    return model

# Quick check
_tmp = build_vit_b16(NUM_CLASSES)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"ViT-B/16  |  Trainable params: {n_params:,}  |  Output logits: {NUM_CLASSES}")
del _tmp

## 7. Entrenamiento — Modelo A (Stratified Split)

In [ ]:
print("=" * 55)
print("  MODELO A — STRATIFIED SPLIT")
print("=" * 55)

L.seed_everything(SEED, workers=True)

backbone_A = build_vit_b16(NUM_CLASSES)
lit_A = LightningModuleMultilabel(
    model        = backbone_A,
    num_classes  = NUM_CLASSES,
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
    max_epochs   = MAX_EPOCHS,
    threshold    = THRESHOLD,
    pos_weight   = pos_w_A,
)

ckpt_A = ModelCheckpoint(
    dirpath   = "outputs/checkpoints",
    filename  = "vitA_stratified-best-{epoch:02d}-{val_f1:.4f}",
    monitor   = "val_f1", mode="max", save_top_k=1, save_weights_only=True,
)
early_A = EarlyStopping(
    monitor   = "val_f1", mode="max",
    patience  = EARLYSTOPPING_EPOCHS, min_delta=1e-3, verbose=True,
)
logger_A = CSVLogger("outputs/logs", name="vitA_stratified")

trainer_A = L.Trainer(
    max_epochs      = MAX_EPOCHS,
    accelerator     = "auto",
    devices         = "auto",
    callbacks       = [ckpt_A, early_A],
    logger          = logger_A,
    log_every_n_steps = 5,
)
trainer_A.fit(lit_A, train_loader_A, val_loader_A)

## 8. Entrenamiento — Modelo B (Random Split)

In [ ]:
print("=" * 55)
print("  MODELO B — RANDOM SPLIT")
print("=" * 55)

L.seed_everything(SEED, workers=True)   # misma seed → misma inicialización de pesos

backbone_B = build_vit_b16(NUM_CLASSES)
lit_B = LightningModuleMultilabel(
    model        = backbone_B,
    num_classes  = NUM_CLASSES,
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
    max_epochs   = MAX_EPOCHS,
    threshold    = THRESHOLD,
    pos_weight   = pos_w_B,
)

ckpt_B = ModelCheckpoint(
    dirpath   = "outputs/checkpoints",
    filename  = "vitB_random-best-{epoch:02d}-{val_f1:.4f}",
    monitor   = "val_f1", mode="max", save_top_k=1, save_weights_only=True,
)
early_B = EarlyStopping(
    monitor   = "val_f1", mode="max",
    patience  = EARLYSTOPPING_EPOCHS, min_delta=1e-3, verbose=True,
)
logger_B = CSVLogger("outputs/logs", name="vitB_random")

trainer_B = L.Trainer(
    max_epochs      = MAX_EPOCHS,
    accelerator     = "auto",
    devices         = "auto",
    callbacks       = [ckpt_B, early_B],
    logger          = logger_B,
    log_every_n_steps = 5,
)
trainer_B.fit(lit_B, train_loader_B, val_loader_B)

## 9. Evaluación en test — Modelo A (Stratified)

In [ ]:
print("=" * 55)
print("  EVALUACIÓN MODELO A — STRATIFIED SPLIT")
print("=" * 55)

trainer_A.test(lit_A, dataloaders=test_loader_A, ckpt_path="best")

best_path_A = ckpt_A.best_model_path
print(f"Best checkpoint: {best_path_A}")
lit_A = LightningModuleMultilabel.load_from_checkpoint(best_path_A, model=lit_A.model)

preds_out_A = trainer_A.predict(lit_A, dataloaders=test_loader_A)
probs_A  = torch.cat([b["probs"]  for b in preds_out_A]).numpy()
preds_A  = torch.cat([b["preds"]  for b in preds_out_A]).numpy()
labels_A = torch.cat([b["labels"] for b in preds_out_A]).numpy()

metrics_A = compute_test_metrics(preds_A, labels_A, probs_A)
print("\nModel A (Stratified) — Test Metrics:")
for k, v in metrics_A.items():
    print(f"  {k:<15}: {v:.4f}")

append_metrics_to_csv(metrics_A, model_name="ViT_B16_Stratified")

## 10. Evaluación en test — Modelo B (Random)

In [ ]:
print("=" * 55)
print("  EVALUACIÓN MODELO B — RANDOM SPLIT")
print("=" * 55)

trainer_B.test(lit_B, dataloaders=test_loader_B, ckpt_path="best")

best_path_B = ckpt_B.best_model_path
print(f"Best checkpoint: {best_path_B}")
lit_B = LightningModuleMultilabel.load_from_checkpoint(best_path_B, model=lit_B.model)

preds_out_B = trainer_B.predict(lit_B, dataloaders=test_loader_B)
probs_B  = torch.cat([b["probs"]  for b in preds_out_B]).numpy()
preds_B  = torch.cat([b["preds"]  for b in preds_out_B]).numpy()
labels_B = torch.cat([b["labels"] for b in preds_out_B]).numpy()

metrics_B = compute_test_metrics(preds_B, labels_B, probs_B)
print("\nModel B (Random) — Test Metrics:")
for k, v in metrics_B.items():
    print(f"  {k:<15}: {v:.4f}")

append_metrics_to_csv(metrics_B, model_name="ViT_B16_Random")

## 11. Tabla comparativa y veredicto de hipótesis

In [ ]:
comparison_df = pd.DataFrame({
    "Metric":     list(metrics_A.keys()),
    "Stratified": [round(v, 4) for v in metrics_A.values()],
    "Random":     [round(v, 4) for v in metrics_B.values()],
})
comparison_df["Δ (Random − Strat)"] = (
    comparison_df["Random"] - comparison_df["Stratified"]
).round(4)

print("\n" + "=" * 55)
print("  STRATIFIED vs RANDOM — COMPARATIVA FINAL")
print("=" * 55)
print(comparison_df.to_string(index=False))

# ── Hypothesis verdict ────────────────────────────────────────────────────────
delta_f1  = abs(metrics_A["macro_f1"] - metrics_B["macro_f1"])
delta_map = abs(metrics_A["macro_map"] - metrics_B["macro_map"])

print(f"\nΔ macro-F1  = {delta_f1:.4f}")
print(f"Δ macro-mAP = {delta_map:.4f}")

if delta_f1 < 0.02:
    print("\n✓ H0 no rechazada: Δ < 2 pp → el pre-entrenamiento compensa el desbalance.")
else:
    print("\n✗ H1 confirmada: Δ ≥ 2 pp → el split estratificado sigue siendo necesario.")

## 12. Curvas de aprendizaje

In [ ]:
plot_training_curves(
    logger_A,
    model_name = "ViT-B/16 — Stratified Split",
    save_path  = "outputs/figures/vitA_stratified_curves.png",
)

plot_training_curves(
    logger_B,
    model_name = "ViT-B/16 — Random Split",
    save_path  = "outputs/figures/vitB_random_curves.png",
)

## 13. Métricas per-class — F1 y AP por clase

In [ ]:
plot_per_class_metrics(
    labels_A, preds_A, probs_A, classes,
    macro_f1   = metrics_A["macro_f1"],
    macro_map  = metrics_A["macro_map"],
    model_name = "ViT-B/16 Stratified",
    save_path  = "outputs/figures/vitA_per_class.png",
    csv_output = "outputs/vitA_per_class.csv",
)

plot_per_class_metrics(
    labels_B, preds_B, probs_B, classes,
    macro_f1   = metrics_B["macro_f1"],
    macro_map  = metrics_B["macro_map"],
    model_name = "ViT-B/16 Random",
    save_path  = "outputs/figures/vitB_per_class.png",
    csv_output = "outputs/vitB_per_class.csv",
)

## 14. Referencias

- Dosovitskiy et al. (2021). *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR 2021.
- Yang et al. (2020). *ML-GCN: Multi-Label Image Recognition with Graph Convolutional Networks*. CVPR.
- UCMerced Land Use Dataset: http://weegee.vision.ucmerced.edu/datasets/landuse.html